In [ ]:
# -*- coding: utf-8 -*-



import re, time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# ===================== PARÂMETROS =====================
REF_TEMP      = 20
FREQ_MIN_KHZ  = 30
FREQ_MAX_KHZ  = 50
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"

# Conjuntos fixos
TEMPS_TREINO = {0, 10, 40, 60}
TEMPS_PROVA  = {-10, 30, 50, 70}

# Banda e compensação
SMOOTH_WIN          = 5
TAU_MAX_FRAC        = 0.025
ANCHOR_TO_REF_ENDS  = True

# Caps de segurança
CAP_GAIN_FRAC   = 0.60
CAP_OFFSET_FRAC = 0.60
CAP_TILT_FRAC   = 0.40

# Park (comparação)
PARK_MAX_SHIFT_FRAC = 0.25
PARK_OVERLAP_MIN    = 0.60
PARK_SMOOTH_WIN     = 5

# ===================== HELPERS BÁSICOS =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win<=1 or win%2==0: return arr
    r=win//2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s/float(win)

def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

# ===================== FEATURES =====================
def spectral_entropy(x):
    ps = np.abs(x)**2
    ps = ps/(np.sum(ps)+1e-12)
    return float(-np.sum(ps*np.log(ps+1e-12)))

def roughness(x):
    return float(np.mean(np.abs(np.diff(x,2))))

def peak_ratio(x):
    idx = np.argpartition(x, -2)[-2:]
    vals = np.sort(x[idx])
    if len(vals)<2 or vals[1]==0: return 0.0
    return float(vals[1]/(vals[0]+1e-12))

def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w  = xm*xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18: return float(np.mean(f))
    num = float(np.trapezoid(f*w, f))
    return num/den

def slope_over_band(f, x):
    return float((x[-1]-x[0])/(f[-1]-f[0] + 1e-12))

def compute_features(X, f):
    """
    Features globais reforçadas:
      mean, std, amp, slope, peak_pos_rel, centroid,
      skew, kurtosis, energias em 5 bandas,
      entropy, roughness, peak_ratio.
    """
    X = np.asarray(X, float); n, m = X.shape
    out=[]
    cuts = np.linspace(f[0], f[-1], 6)  # 5 bandas
    for i in range(n):
        x = X[i]
        mean  = float(np.mean(x))
        std   = float(np.std(x))
        amp   = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        pk_i  = int(np.argmax(x)); peak_pos_rel = pk_i / max(1,(m-1))
        centroid = energy_weighted_centroid(f, x)
        z = (x - mean)/(std + 1e-12)
        skew = float(np.mean(z**3))
        kurt = float(np.mean(z**4))
        # energias
        E_bands=[]
        for j in range(len(cuts)-1):
            mask = (f>=cuts[j]) & (f<cuts[j+1])
            if mask.sum()<2: E_bands.append(0.0)
            else: E_bands.append(float(np.trapezoid((x[mask]**2), f[mask])))
        ent  = spectral_entropy(x)
        rough= roughness(x)
        pr   = peak_ratio(x)
        out.append([mean,std,amp,slope,peak_pos_rel,centroid,skew,kurt,*E_bands,ent,rough,pr])
    cols = ["mean","std","amp","slope","peak_pos_rel","centroid","skew","kurt",
            "E_b1","E_b2","E_b3","E_b4","E_b5","entropy","roughness","peak_ratio"]
    return np.array(out, float), cols

def fit_feature_vs_temp_models(F, T, names):
    models = {}
    T = np.asarray(T, float).reshape(-1,1)
    for j, name in enumerate(names):
        lr = LinearRegression().fit(T, F[:,j])
        models[name] = lr
    return models

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(lr.predict(Tref)[0]) for name,lr in models.items()}

# ===================== COMPENSAÇÃO POR VARIÁVEIS =====================
def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    x = x.copy()
    mean_t = targets["mean"]
    amp_t  = targets["amp"]
    slope_t= targets["slope"]
    centroid_t = targets.get("centroid", None)

    mean_x = float(x.mean())
    amp_x  = float(x.max() - x.min())
    slope_x= slope_over_band(f, x)

    # 1) offset
    offset = mean_t - mean_x
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))
    x = x + offset

    # 2) ganho
    gain = 1.0 if amp_x<=1e-9 else float(amp_t/amp_x)
    gmin = 1.0 - caps["gain_frac"]; gmax = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, gmin, gmax))
    x = mean_t + gain*(x - mean_t)

    # 3) tilt
    delta_slope = slope_t - slope_x
    u = np.linspace(-0.5, 0.5, len(x))
    df = (f[-1]-f[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)
    x = x + tilt_signal

    # 4) micro-shift (aproxima centroid)
    if centroid_t is not None:
        cent_x = energy_weighted_centroid(f, x)
        delta_c = centroid_t - cent_x
        tau_max = TAU_MAX_FRAC * (f[-1]-f[0])
        tau = float(np.clip(delta_c, -tau_max, tau_max))
        if abs(tau) > 1e-12:
            x = shift_interp(x, f, tau)

    # 5) âncora nos extremos
    if ANCHOR_TO_REF_ENDS and (y_ref is not None):
        e0 = x[0]   - y_ref[0]
        e1 = x[-1]  - y_ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x = x - corr

    return x

def compensate_set_by_features(X, f, feat_models, ref_temp, y_ref, caps, smooth_win=SMOOTH_WIN):
    targets = feature_targets_at_ref(feat_models, ref_temp)
    Y = np.zeros_like(X)
    for i in range(X.shape[0]):
        yi = apply_compensation_by_features(X[i], f, targets, caps, y_ref=y_ref)
        if smooth_win>1 and (smooth_win%2==1):
            yi = moving_average(yi, smooth_win)
        Y[i] = yi
    return Y, targets

# ===================== PARK (1999) – comparação) =====================
def park_compensate_single(x, y_ref, fhz,
                           max_shift_frac=PARK_MAX_SHIFT_FRAC,
                           overlap_min_frac=PARK_OVERLAP_MIN,
                           smooth_win=PARK_SMOOTH_WIN):
    """
    Implementação fiel ao Park (1999) descrita no TCC de Dias:
    - Varre deslocamento contínuo δτ (até max_shift_frac da banda).
    - Ajusta δS = média(Y_ref - X_shift) no overlap.
    - Minimiza Va = soma((Y_ref - (X_shift+δS))^2).
    """
    n = len(x)
    fmin, fmax = fhz[0], fhz[-1]
    df_band = fmax - fmin

    # faixa de busca para δτ (em Hz)
    tau_max = max_shift_frac * df_band
    nsteps = 101  # resolução da busca
    tau_vals = np.linspace(-tau_max, tau_max, nsteps)

    best = (np.inf, 0.0, 0.0)  # (Va, δτ, δS)

    for tau in tau_vals:
        # desloca curva por δτ (interpolação)
        x_shift = shift_interp(x, fhz, tau)

        # define overlap (mesmo comprimento)
        xs = x_shift
        yr = y_ref
        if len(xs) < int(overlap_min_frac*n):
            continue

        # offset δS
        deltaS = float(np.mean(yr - xs))

        # energia residual
        resid = yr - (xs + deltaS)
        Va = float(np.sum(resid*resid))

        if Va < best[0]:
            best = (Va, tau, deltaS)

    _, tau_best, dS_best = best

    # aplica melhor shift+offset encontrado
    yout = shift_interp(x, fhz, tau_best) + dS_best

    # suavização opcional
    if smooth_win > 1 and smooth_win % 2 == 1:
        yout = moving_average(yout, smooth_win)

    return yout, tau_best, dS_best


def park_batch(X, y_ref, fhz):
    n, m = X.shape
    Y = np.zeros_like(X)
    taus, deltas = [], []
    for i in range(n):
        yi, tau, dS = park_compensate_single(X[i], y_ref, fhz)
        Y[i] = yi
        taus.append(tau)
        deltas.append(dS)
    return Y, np.array(taus), np.array(deltas)


# ===================== CARGA =====================
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

freq_cols_tr, _ = get_freq_columns(base_tr, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
freq_cols_te, _ = get_freq_columns(base_te, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
order = np.argsort(fhz); common_cols = [common_cols[i] for i in order]; fhz = fhz[order]
fkHz = fhz/1e3

X_tr_full = base_tr[common_cols].to_numpy(float)
X_te_full = base_te[common_cols].to_numpy(float)
T_tr_full = base_tr["temp_c"].to_numpy(float)
T_te_full = base_te["temp_c"].to_numpy(float)

pool_20=[]
if (base_tr["temp_c"]==REF_TEMP).any():
    pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
if (base_te["temp_c"]==REF_TEMP).any():
    pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
assert len(pool_20)>0, "Não há curva real @20°C!"
y_ref = np.median(np.vstack(pool_20), axis=0)

# restrições de treino/prova
tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()

X_tr = tr_restr[common_cols].to_numpy(float)
X_te = te_restr[common_cols].to_numpy(float)
T_tr = tr_restr["temp_c"].to_numpy(float)
T_te = te_restr["temp_c"].to_numpy(float)

# ===================== RF-Temp (duplo) =====================
F_tr, feat_names = compute_features(X_tr, fhz)
F_te, _          = compute_features(X_te, fhz)

# RF restrito (importâncias)
rf_temp_restr = RandomForestRegressor(
    n_estimators=3000, max_depth=40,
    max_samples=0.9, n_jobs=-1, random_state=42
).fit(F_tr, T_tr)

print("\n== RF-Temp (restrito: só treino fixo) ==")
print(f"R²(treino) = {rf_temp_restr.score(F_tr, T_tr):.3f}")
print("Importâncias (restrito):")
for name, imp in sorted(zip(feat_names, rf_temp_restr.feature_importances_), key=lambda x:-x[1]):
    print(f"{name:>12s}: {imp:.3f}")

# RF global (previsão de T)
F_full, feat_names_full = compute_features(np.vstack([X_tr, X_te]), fhz)
T_full = np.concatenate([T_tr, T_te])

rf_temp_global = RandomForestRegressor(
    n_estimators=3000, max_depth=40,
    max_samples=0.9, n_jobs=-1, random_state=123
).fit(F_full, T_full)

print("\n== RF-Temp GLOBAL (treino+prova) ==")
print(f"R²(treino+prova) = {rf_temp_global.score(F_full, T_full):.3f}")

# ===================== MODELOS feature~T E COMPENSAÇÃO =====================
feat_models = fit_feature_vs_temp_models(F_tr, T_tr, feat_names)
feat_targets_20 = feature_targets_at_ref(feat_models, REF_TEMP)
print("\nTargets de features em 20°C (via regressão no treino):")
for k,v in feat_targets_20.items():
    print(f"{k:>12s}: {v:.6f}")

caps = dict(gain_frac=CAP_GAIN_FRAC, offset_frac=CAP_OFFSET_FRAC, tilt_frac=CAP_TILT_FRAC)
t0 = time.time()
Y_te_hat, _targets = compensate_set_by_features(X_te, fhz, feat_models, REF_TEMP, y_ref, caps, smooth_win=SMOOTH_WIN)
print(f"[INFO] Compensação por features aplicada em {len(X_te)} curvas em {time.time()-t0:.2f}s")

# ===================== PARK (comparação) =====================
Y_te_park, park_taus, park_deltas = park_batch(X_te, y_ref, fhz)

# ===================== Checagem RF-Temp nas curvas finais =====================
F_final, _ = compute_features(Y_te_hat, fhz)
T_hat_final = rf_temp_global.predict(F_final)
print("\n### Checagem RF-Temp nas curvas finais ###")
print(f"média={float(np.mean(T_hat_final)):.2f}°C | desvio={float(np.std(T_hat_final)):.2f}°C | "
      f"MAE vs {REF_TEMP}°C={float(np.mean(np.abs(T_hat_final-REF_TEMP))):.2f}°C")

# ===================== PLOTS (exemplos) =====================
def _prep_plot():
    plt.rcParams.update({
        "figure.figsize": (9.2, 5.0),
        "axes.grid": True, "grid.alpha": 0.28,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.labelsize": 12, "axes.titlesize": 13,
        "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 10, "lines.linewidth": 1.8,
    })

def plot_rf(i=0, save=False, prefix="rf_comp"):
    _prep_plot()
    fhz_khz = fkHz
    T_real = float(te_restr.iloc[i]["temp_c"])
    F_orig, _ = compute_features(X_te[i][None,:], fhz)
    F_comp, _ = compute_features(Y_te_hat[i][None,:], fhz)
    T_pred_orig  = float(rf_temp_global.predict(F_orig)[0])
    T_pred_final = float(rf_temp_global.predict(F_comp)[0])

    fig, ax = plt.subplots()
    ax.plot(fhz_khz, X_te[i],     label=f"Original @ {T_real:.0f} °C (RF °C)")
    ax.plot(fhz_khz, y_ref,       label=f"Referência @ {REF_TEMP} °C")
    ax.plot(fhz_khz, Y_te_hat[i], label=f"Compensada (RF°C)")
    ax.set_title(f"Amostra {i} — Compensação por variáveis")
    ax.set_xlabel("Frequência (kHz)"); ax.set_ylabel("Re{Z}")
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    if save: fig.savefig(f"{prefix}_i{i}.png", dpi=300)
    plt.show()

def plot_park(i=0, save=False, prefix="park_comp"):
    _prep_plot()
    fhz_khz = fkHz
    T_real = float(te_restr.iloc[i]["temp_c"])
    fig, ax = plt.subplots()
    ax.plot(fhz_khz, X_te[i],      label=f"Original @ °C")
    ax.plot(fhz_khz, y_ref,        label=f"Referência @  °C")
    ax.plot(fhz_khz, Y_te_park[i], label=f"Park (comp.)")
    ax.set_title(f"Amostra {i} — Park (1999) – comparação")
    ax.set_xlabel("Frequência (kHz)"); ax.set_ylabel("Re{Z}")
    ax.legend(loc="best", frameon=False)
    fig.tight_layout()
    if save: fig.savefig(f"{prefix}_i{i}.png", dpi=300)
    plt.show()

# Exemplos de plot:
plot_rf(i=0, save=False)
plot_park(i=0, save=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# === Configurações gerais ===
plt.rcParams.update({
    'font.size': 20,
    'text.usetex': True,
    'font.family': 'Times New Roman'
})

def plot_compensacao_rf(fhz_khz, X_te, y_ref, Y_te_hat, i, T_real, REF_TEMP=20, prefix='RF_comp', save=True):
    """
    Gera o gráfico no estilo 'Force × yc' comparando:
      - Curva original @ T_real °C
      - Referência @ REF_TEMP °C
      - Curva compensada (RF)
    """
    fig, ax = plt.subplots(figsize=(7, 5))

    # === Plot principal ===
    ax.plot(fhz_khz, X_te[i], '-', lw=2.5, color='tab:red',
            label=rf'\textbf{{Original @ {T_real:.0f}$^\circ$C}}')
    ax.plot(fhz_khz, y_ref, '--', lw=2.5, color='k',
            label=rf'\textbf{{Referência @ {REF_TEMP}$^\circ$C}}')
    ax.plot(fhz_khz, Y_te_hat[i], '-', lw=2.5, color='tab:blue',
            label=rf'\textbf{{Compensada (RF)}}')

    # === Formatação ===
    ax.set_xlabel(r'Frequência [kHz]', fontsize=20, rotation=0, labelpad=12)
    ax.set_ylabel(r'$\mathrm{Re}\{Z\}$ [a.u.]', fontsize=20, rotation=90, labelpad=18)
    ax.set_title(rf'\textbf{{Amostra {i} --- Compensação por variáveis}}', fontsize=20, pad=15)

    # Limites automáticos, mas com borda de 2%
    fmin, fmax = fhz_khz.min(), fhz_khz.max()
    ax.set_xlim([fmin - 0.02*(fmax-fmin), fmax + 0.02*(fmax-fmin)])

    # Grade e estilo
    ax.grid(True, ls=':', lw=0.7, alpha=0.7)
    ax.legend(frameon=False, fontsize=16, loc='best')

    # Layout e salvamento
    fig.tight_layout()
    if save:
        nomefig = f'{prefix}_amostra{i}_T{int(T_real)}C.pdf'
        fig.savefig(nomefig, bbox_inches='tight', transparent=True)
        print(f"Figura salva como: {nomefig}")

    plt.show()


In [ ]:
# Exemplo hipotético
i = 0
T_real = 46
fhz_khz = fhz / 1e3
plot_compensacao_rf(fhz_khz, X_te, y_ref, Y_te_hat, i, T_real, REF_TEMP=20, prefix='Faixa35_75')
